# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsanalee/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** Flag content sitting on page 2 of search (average position 11–20) as a *quick-win* candidate, because it is close enough to page 1 that a small ranking improvement could unlock clicks from impressions the content is already earning. Among page-2 content, rank by how much existing impression volume (`imp_prev30`) is at stake — the more volume already there, the bigger the payoff from closing the gap.

**Reason codes this rule can output:**
- `QUICK_WIN_PAGE2_VOLUME` — page-2 position (11–20) with impression volume behind it.
- `NO_ACTION` — outside the eligible position band; no rule fires.

**Why these two signals:** `avg_position_prev30` decides *eligibility* (is this content close enough to the front page to matter), and `imp_prev30` decides *priority within* that eligible set (is anyone actually seeing it). Both are pre-decision, previous-30-day signals from the Week 3 data contract — no outcome-window or label-derived fields are used.

## 1. Signal checks

I check two pre-decision signals before encoding the baseline rule. The first is `imp_prev30` (volume), which is linked to the quick-win logic. The second is `avg_position_prev30`. Both checks use only pre-decision signals from the Week 3 data contract.

**Signal check A — `avg_position_prev30` vs CTR.** Standard SEO expectation: click-through rate falls as average position gets worse. I bucket position into ranges and print `n` and mean CTR (`clicks_prev30 / imp_prev30`) per bucket. If CTR clearly declines as position worsens, that's `CONFIRMED` and it justifies treating position as the eligibility gate for quick-win (page-2 content still gets meaningfully fewer clicks than page-1 content, so there's real room to gain).

**Signal check B — `imp_prev30` inside the page-2 band (positions 11–20).** The quick-win rule only matters if, within that eligible band, volume actually varies enough to be worth ranking on. I bucket `imp_prev30` into quartiles *restricted to the 11–20 position band* and print `n` and mean impressions/clicks per bucket. If the buckets show a real, ordered spread (not flat, not reversed), that's `CONFIRMED` — volume is a meaningful tie-breaker inside the band.

In [1]:
# ---- Signal check A: avg_position_prev30 vs CTR ----
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
""")

from huggingface_hub import snapshot_download
from pathlib import Path

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=["fact_content_daily_performance/month=2026-03/*"],
)
path_prev = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=["fact_content_daily_performance/month=2026-02/*"],
)

march_files = list(Path(path).rglob("fact_content_daily_performance/month=2026-03/*.parquet"))
feb_files = list(Path(path_prev).rglob("fact_content_daily_performance/month=2026-02/*.parquet"))

print("March files:", len(march_files), "| February files:", len(feb_files))

# Build the same prev30 feature frame as the Week 3 contract (Mar 2 - Mar 31 window).
features = con.sql(f"""
    WITH daily AS (
        SELECT * FROM read_parquet(['{feb_files[0]}', '{march_files[0]}'])
    ),
    prev30 AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(COALESCE(gsc_impressions, 0)) AS imp_prev30,
            SUM(COALESCE(gsc_clicks, 0)) AS clicks_prev30,
            AVG(gsc_avg_position) AS avg_position_prev30
        FROM daily
        WHERE report_date >= DATE '2026-03-02'
          AND report_date <= DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT * FROM prev30 WHERE imp_prev30 > 0
""").df()

query_path = Path(path) / "fact_content_query_90d.parquet"
query_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(content_visible_query_count) AS visible_queries,
        MAX(anonymized_impressions_share) AS top_query_share
    FROM read_parquet('{query_path}')
    GROUP BY 1, 2
""").df()

feature_frame = features.merge(query_features, on=["client_hash_id", "content_hash_id"], how="left")
feature_frame["visible_queries"] = feature_frame["visible_queries"].fillna(0)
feature_frame["top_query_share"] = feature_frame["top_query_share"].fillna(0)
feature_frame["ctr_prev30"] = feature_frame["clicks_prev30"] / feature_frame["imp_prev30"]

print("feature_frame rows:", len(feature_frame))
feature_frame.head()

c:\Users\meerm\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 478.26it/s]


March files: 1 | February files: 1
feature_frame rows: 176268


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,visible_queries,top_query_share,ctr_prev30
0,client_73cda7b4e4f265ea,content_1c5b788f459efc0f,362.0,2.0,2.839288,5.0,0.690711,0.005525
1,client_73cda7b4e4f265ea,content_f2247734067613ef,278.0,0.0,5.739229,3.0,0.718593,0.000000
2,client_73cda7b4e4f265ea,content_c892bc4727c915fd,29.0,0.0,7.235714,0.0,0.000000,0.000000
3,client_73cda7b4e4f265ea,content_d1e5d0a7df416b6b,221.0,0.0,33.252709,16.0,0.465134,0.000000
4,client_73cda7b4e4f265ea,content_98c2734f085edb92,177.0,0.0,6.202968,0.0,0.000000,0.000000


In [2]:
# ---- Bucket table: avg_position_prev30 vs CTR ----
position_bins = [0, 3, 10, 20, 50, float("inf")]
position_labels = ["1-3", "4-10", "11-20", "21-50", "51+"]

feature_frame["position_bucket"] = pd.cut(
    feature_frame["avg_position_prev30"], bins=position_bins, labels=position_labels
)

position_table = feature_frame.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr_prev30", "mean"),
    mean_imp=("imp_prev30", "mean"),
).reset_index()

print("Signal check A - avg_position_prev30 vs CTR")
print(position_table.to_string(index=False))

# Verdict: does CTR fall as position worsens across the ordered buckets?
ctr_values = position_table["mean_ctr"].tolist()
is_monotonic_decline = all(ctr_values[i] >= ctr_values[i + 1] for i in range(len(ctr_values) - 1))
print()
print("VERDICT (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE")
print("Monotonic decline across ALL buckets:", is_monotonic_decline)

Signal check A - avg_position_prev30 vs CTR
position_bucket     n  mean_ctr    mean_imp
            1-3 16057  0.010582 2370.862864
           4-10 81513  0.004921 1754.821047
          11-20 32109  0.003235 1054.216637
          21-50 33332  0.002209 1677.366045
            51+ 11800  0.000924  179.096949

VERDICT (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE
Monotonic decline across ALL buckets: True


In [3]:
# ---- Bucket table: imp_prev30 within the page-2 band (11-20), signal behind quick-win logic ----
page2 = feature_frame[feature_frame["avg_position_prev30"].between(11, 20)].copy()
print("Rows in the 11-20 position band (n):", len(page2))

page2["imp_quartile"] = pd.qcut(page2["imp_prev30"], q=4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])

volume_table = page2.groupby("imp_quartile", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_imp=("imp_prev30", "mean"),
    mean_clicks=("clicks_prev30", "mean"),
    mean_ctr=("ctr_prev30", "mean"),
).reset_index()

print()
print("Signal check B - imp_prev30 quartiles inside the 11-20 position band")
print(volume_table.to_string(index=False))
print()
print("VERDICT (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE")

Rows in the 11-20 position band (n): 27379

Signal check B - imp_prev30 quartiles inside the 11-20 position band
imp_quartile    n    mean_imp  mean_clicks  mean_ctr
 Q1 (lowest) 6854   17.017800     0.058652  0.006014
          Q2 6837  134.066257     0.292526  0.002244
          Q3 6843  494.194359     0.998539  0.002009
Q4 (highest) 6845 3640.546384    11.689847  0.002858

VERDICT (fill in after reading the table): CONFIRMED / OPPOSITE / MIXED / FALSE


**Verdicts (fill in after running the two cells above with real numbers):**

- Signal check A (`avg_position_prev30` vs CTR): **[CONFIRMED / OPPOSITE / MIXED / FALSE]** — one sentence on what the bucket table showed and why that verdict, referencing the printed `n`.
- Signal check B (`imp_prev30` inside the page-2 band): **[CONFIRMED / OPPOSITE / MIXED / FALSE]** — one sentence on what the bucket table showed and why that verdict, referencing the printed `n`.

*A clearly-explained negative here (OPPOSITE/MIXED/FALSE) is still a valid outcome — write honestly about what the numbers show, even if it means adjusting the rule below.*

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

**Score:** for content in the 11–20 position band, `score = imp_prev30` (raw volume already being seen). Outside that band, `score = 0`.
**Reason code:** `QUICK_WIN_PAGE2_VOLUME` when eligible, `NO_ACTION` otherwise.
**Action label:** `REVIEW_QUICK_WIN` when eligible, `NO_ACTION` otherwise.

In [4]:
# ---- Encode the rule and write the ranked queue ----
import os

def score_row(row):
    if 11 <= row["avg_position_prev30"] <= 20:
        return row["imp_prev30"], "QUICK_WIN_PAGE2_VOLUME", "REVIEW_QUICK_WIN"
    return 0.0, "NO_ACTION", "NO_ACTION"

scored = feature_frame.copy()
scored[["score", "reason_code", "action_label"]] = scored.apply(
    lambda r: pd.Series(score_row(r)), axis=1
)

ranked_queue = scored.sort_values("score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

output_cols = [
    "rank", "client_hash_id", "content_hash_id",
    "imp_prev30", "clicks_prev30", "avg_position_prev30", "ctr_prev30",
    "score", "reason_code", "action_label",
]

os.makedirs("work/outputs", exist_ok=True)
ranked_queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Wrote work/outputs/baseline_action_score.csv")
print("Total rows:", len(ranked_queue))
print("Eligible (REVIEW_QUICK_WIN) rows:", (ranked_queue["action_label"] == "REVIEW_QUICK_WIN").sum())
ranked_queue[output_cols].head(10)

Wrote work/outputs/baseline_action_score.csv
Total rows: 176268
Eligible (REVIEW_QUICK_WIN) rows: 27379


,rank,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,ctr_prev30,score,reason_code,action_label
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,239235.0,647.0,14.968027,0.002704,239235.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
1,2,client_23a62021009f63c4,content_66288edeb93b7c4f,137424.0,780.0,18.312478,0.005676,137424.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
2,3,client_23a62021009f63c4,content_5e1c049f62e33b11,117698.0,166.0,18.113831,0.001410,117698.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
3,4,client_23a62021009f63c4,content_f6723f0229e1bfdc,68080.0,13.0,15.690774,0.000191,68080.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
4,5,client_23a62021009f63c4,content_65c75874a23fca87,64743.0,18.0,13.505655,0.000278,64743.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
5,6,client_23a62021009f63c4,content_2690f62f39fb14fe,59743.0,97.0,17.613134,0.001624,59743.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,54887.0,1.0,11.568483,0.000018,54887.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
7,8,client_23a62021009f63c4,content_1df00c7789a0adcc,52759.0,130.0,18.243727,0.002464,52759.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
8,9,client_23a62021009f63c4,content_5be6be2550a98fc5,52201.0,160.0,16.750697,0.003065,52201.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
9,10,client_20259bd6705d81d4,content_653bbcddf2314227,50068.0,20.0,18.723232,0.000399,50068.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN


## 3. Top-10 review

*For each of the top 10: action, why it's there, and what would make it wrong.*

In [5]:
# ---- Print the top 10 rows for the review below ----
top10 = ranked_queue[output_cols].head(10)
top10

,rank,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,ctr_prev30,score,reason_code,action_label
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,239235.0,647.0,14.968027,0.002704,239235.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
1,2,client_23a62021009f63c4,content_66288edeb93b7c4f,137424.0,780.0,18.312478,0.005676,137424.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
2,3,client_23a62021009f63c4,content_5e1c049f62e33b11,117698.0,166.0,18.113831,0.001410,117698.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
3,4,client_23a62021009f63c4,content_f6723f0229e1bfdc,68080.0,13.0,15.690774,0.000191,68080.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
4,5,client_23a62021009f63c4,content_65c75874a23fca87,64743.0,18.0,13.505655,0.000278,64743.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
5,6,client_23a62021009f63c4,content_2690f62f39fb14fe,59743.0,97.0,17.613134,0.001624,59743.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
6,7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,54887.0,1.0,11.568483,0.000018,54887.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
7,8,client_23a62021009f63c4,content_1df00c7789a0adcc,52759.0,130.0,18.243727,0.002464,52759.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
8,9,client_23a62021009f63c4,content_5be6be2550a98fc5,52201.0,160.0,16.750697,0.003065,52201.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN
9,10,client_20259bd6705d81d4,content_653bbcddf2314227,50068.0,20.0,18.723232,0.000399,50068.0,QUICK_WIN_PAGE2_VOLUME,REVIEW_QUICK_WIN


Fill in one line per row using the printed `top10` table above (rank, action, why it's there, what would make it wrong):
Rank 1 — Action: REVIEW_QUICK_WIN. Why: content_e8a52cf3d5988c07 has 239,235 impressions at position 14.97 — by far the largest volume in the 11–20 band. Would be wrong if: the position average is being pulled down by a couple of anomalous days rather than a stable page-2 ranking.
Rank 2 — Action: REVIEW_QUICK_WIN. Why: content_66288edeb93b7c4f has 137,424 impressions at position 18.31, with the best CTR in the top 10 (0.57%), suggesting real click intent already exists. Would be wrong if: that CTR is driven by a handful of days rather than being consistent across the window.
Rank 3 — Action: REVIEW_QUICK_WIN. Why: content_5e1c049f62e33b11 has 117,698 impressions at position 18.11. Would be wrong if: the queries driving those impressions are only loosely related to the content (low relevance), which would cap any gain from a ranking improvement.
Rank 4 — Action: REVIEW_QUICK_WIN. Why: content_f6723f0229e1bfdc has 68,080 impressions at position 15.69. Would be wrong if: CTR here is only 0.02% (13 clicks on 68k impressions) — that's low even for page 2, hinting the volume may be low-intent or mismatched queries rather than a true quick win.
Rank 5 — Action: REVIEW_QUICK_WIN. Why: content_65c75874a23fca87 has 64,743 impressions at position 13.51 — closest to page 1 in the top 10. Would be wrong if: the low CTR (0.03%, 18 clicks) means the ranking is already about as good as intent allows, so a position bump wouldn't move clicks much.
Rank 6 — Action: REVIEW_QUICK_WIN. Why: content_2690f62f39fb14fe has 59,743 impressions at position 17.61 with a moderate CTR (0.16%). Would be wrong if: the content is scheduled for consolidation/redirect elsewhere in the roadmap, making the investment pointless regardless of ranking.
Rank 7 — Action: REVIEW_QUICK_WIN. Why: content_9c057b66c30a3abb has 54,887 impressions at position 11.57 — the best position in the top 10. Would be wrong if: only 1 click came from those 54,887 impressions (CTR 0.002%) — that's an extreme outlier and likely means the impressions are largely irrelevant queries, not real quick-win potential.
Rank 8 — Action: REVIEW_QUICK_WIN. Why: content_1df00c7789a0adcc has 52,759 impressions at position 18.24 with a solid CTR (0.25%). Would be wrong if: this client already has several higher-scoring items above it (ranks 1, 2, 3, 5, 6, 9 are the same client), so a per-client cap might be needed to avoid one client dominating the queue.
Rank 9 — Action: REVIEW_QUICK_WIN. Why: content_5be6be2550a98fc5 has 52,201 impressions at position 16.75 with the second-best CTR in the top 10 (0.31%). Would be wrong if: this is another instance of the same client crowding out other clients' genuine quick wins.
Rank 10 — Action: REVIEW_QUICK_WIN. Why: content_653bbcddf2314227 has 50,068 impressions at position 18.72 — the first item from a different client (client_20259bd6705d81d4). Would be wrong if: the very low CTR (0.04%, 20 clicks) again points to low-relevance impressions rather than genuine unmet demand.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# ---- Leakage check ----
feature_cols_used = ["imp_prev30", "avg_position_prev30"]
print("Feature columns used by the rule:", feature_cols_used)
print("These are pre-decision, previous-30-day signals only (Mar 2 - Mar 31 window).")
print("No outcome-window field (e.g. imp_next30) and no label-derived field (e.g. is_declining,")
print("trend_direction, trend_pct) is referenced anywhere in score_row().")

# Sanity check: confirm none of the excluded/label columns exist in the scored frame.
excluded = {"is_declining", "trend_direction", "trend_pct", "imp_next30", "label"}
present = excluded.intersection(set(scored.columns))
print("Excluded/leaky columns present in scored frame (should be empty):", present)

Feature columns used by the rule: ['imp_prev30', 'avg_position_prev30']
These are pre-decision, previous-30-day signals only (Mar 2 - Mar 31 window).
No outcome-window field (e.g. imp_next30) and no label-derived field (e.g. is_declining,
trend_direction, trend_pct) is referenced anywhere in score_row().
Excluded/leaky columns present in scored frame (should be empty): set()


**Weak picks:** [After reading `top10`, note any rows where the rule fires but the pick looks weak — e.g. `avg_position_prev30` sitting right at the 11 or 20 boundary from a thin sample, or very low `clicks_prev30` despite high `imp_prev30` suggesting a mismatch between impressions and real intent.]

**Leakage confirmation:** The rule uses only `imp_prev30` and `avg_position_prev30`, both previous-30-day, pre-decision fields per the Week 3 contract. The check above confirms no outcome-window or label-derived column (`is_declining`, `trend_direction`, `trend_pct`, future-window impressions) is present in the scored frame or referenced by `score_row()`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.